# 인지·감성 디지털 트윈

한국의 AI 규제 관련 유튜브 댓글과 미국 Meta의 2025년 실적 발표문을 비교하고, 규칙 기반 감성·인지 상태와 마르코프 전이를 분석합니다.

## 실행 안내

1. 저장소 루트에서 JupyterLab을 시작합니다.
2. `data/`에 `Q1_2025.txt`, `Q2_2025.txt`, `Q3_2025.txt`를 준비합니다.
3. 이 Notebook을 위에서 아래로 실행합니다.

유튜브 댓글은 실행 시점에 수집되므로 결과가 달라질 수 있습니다. 데이터와 해석의 한계는 프로젝트 README를 참고하세요.


## 1. 분석 환경 설정


In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yt_dlp
from ipywidgets import FloatSlider, interact
from konlpy.tag import Okt
from textblob import TextBlob
from wordcloud import WordCloud

DATA_DIR = Path("data")


### 한글 폰트 설정

Linux/Colab의 나눔바른고딕 또는 Windows의 맑은 고딕을 우선 사용합니다. 폰트가 없으면 기본 폰트로 실행되며 한글이 깨질 수 있습니다.


In [ ]:
# 실행 환경에서 사용 가능한 한글 폰트를 선택합니다.
font_candidates = [
    Path("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"),
    Path("C:/Windows/Fonts/malgun.ttf"),
]
font_file = next((path for path in font_candidates if path.exists()), None)
font_path = str(font_file) if font_file else None
fontprop = fm.FontProperties(fname=font_path) if font_path else fm.FontProperties()

if font_file:
    plt.rc("font", family=fontprop.get_name())
plt.rcParams["axes.unicode_minus"] = False


## 2. 한국 유튜브 댓글 수집

동일한 수집 로직을 네 개 영상에 순서대로 적용합니다. 각 셀은 `ko_df`와 `data/yt_comments_ko.csv`를 덮어쓰므로 이후 분석은 마지막으로 실행한 영상의 댓글을 사용합니다.


### 2.1 SBS 뉴스 영상


In [ ]:
# 2. 분석할 유튜브 영상 URL (SBS 뉴스 예시)
video_url = "https://www.youtube.com/watch?v=BsrvEziQsdY&start=21" # 여기에 분석할 주소를 넣으세요.

# 3. 댓글 추출 설정
ydl_opts = {
    'getcomments': True,
    'quiet': True,
    'extract_flat': True,
    'skip_download': True,
}

print("댓글 수집 중... 잠시만 기다려주세요.")

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(video_url, download=False)
    comments_data = info.get('comments', [])

# 4. 데이터프레임으로 변환 및 저장
comments_list = []
for c in comments_data:
    comments_list.append({
        'author': c.get('author'),
        'text': c.get('text'),
        'like_count': c.get('like_count'),
        'timestamp': c.get('timestamp')
    })

ko_df = pd.DataFrame(comments_list)
ko_df.to_csv(DATA_DIR / "yt_comments_ko.csv", index=False, encoding='utf-8-sig')

# 5. '처벌' 키워드가 들어간 댓글만 필터링해서 보기
punish_comments = ko_df[ko_df['text'].str.contains('처벌|구속|수사', na=False)]
print(f"총 {len(ko_df)}개의 댓글 중 규제/처벌 관련 댓글 {len(punish_comments)}개를 찾았습니다.")
print(punish_comments['text'].head())


### 2.2 채널A 뉴스 영상 1


In [ ]:
# 2. 분석할 유튜브 영상 URL (채널A 뉴스 예시)
video_url = "https://www.youtube.com/watch?v=jzXAm6Fd_To" # 여기에 분석할 주소를 넣으세요.

# 3. 댓글 추출 설정
ydl_opts = {
    'getcomments': True,
    'quiet': True,
    'extract_flat': True,
    'skip_download': True,
}

print("댓글 수집 중... 잠시만 기다려주세요.")

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(video_url, download=False)
    comments_data = info.get('comments', [])

# 4. 데이터프레임으로 변환 및 저장
comments_list = []
for c in comments_data:
    comments_list.append({
        'author': c.get('author'),
        'text': c.get('text'),
        'like_count': c.get('like_count'),
        'timestamp': c.get('timestamp')
    })

ko_df = pd.DataFrame(comments_list)
ko_df.to_csv(DATA_DIR / "yt_comments_ko.csv", index=False, encoding='utf-8-sig')

# 5. '처벌' 키워드가 들어간 댓글만 필터링해서 보기
punish_comments = ko_df[ko_df['text'].str.contains('처벌|구속|수사', na=False)]
print(f"총 {len(ko_df)}개의 댓글 중 규제/처벌 관련 댓글 {len(punish_comments)}개를 찾았습니다.")
print(punish_comments['text'].head())


### 2.3 채널A 뉴스 영상 2


In [ ]:
# 2. 분석할 유튜브 영상 URL (채널A 뉴스 예시)
video_url = "https://www.youtube.com/watch?v=ipCH7lwM7hg" # 여기에 분석할 주소를 넣으세요.

# 3. 댓글 추출 설정
ydl_opts = {
    'getcomments': True,
    'quiet': True,
    'extract_flat': True,
    'skip_download': True,
}

print("댓글 수집 중... 잠시만 기다려주세요.")

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(video_url, download=False)
    comments_data = info.get('comments', [])

# 4. 데이터프레임으로 변환 및 저장
comments_list = []
for c in comments_data:
    comments_list.append({
        'author': c.get('author'),
        'text': c.get('text'),
        'like_count': c.get('like_count'),
        'timestamp': c.get('timestamp')
    })

ko_df = pd.DataFrame(comments_list)
ko_df.to_csv(DATA_DIR / "yt_comments_ko.csv", index=False, encoding='utf-8-sig')

# 5. '처벌' 키워드가 들어간 댓글만 필터링해서 보기
punish_comments = ko_df[ko_df['text'].str.contains('처벌|구속|수사', na=False)]
print(f"총 {len(ko_df)}개의 댓글 중 규제/처벌 관련 댓글 {len(punish_comments)}개를 찾았습니다.")
print(punish_comments['text'].head())


### 2.4 YTN 뉴스 영상


In [ ]:
# 2. 분석할 유튜브 영상 URL (ytn 뉴스 예시)
video_url = "https://www.youtube.com/watch?v=wPZBpvkP_8A" # 여기에 분석할 주소를 넣으세요.

# 3. 댓글 추출 설정
ydl_opts = {
    'getcomments': True,
    'quiet': True,
    'extract_flat': True,
    'skip_download': True,
}

print("댓글 수집 중... 잠시만 기다려주세요.")

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(video_url, download=False)
    comments_data = info.get('comments', [])

# 4. 데이터프레임으로 변환 및 저장
comments_list = []
for c in comments_data:
    comments_list.append({
        'author': c.get('author'),
        'text': c.get('text'),
        'like_count': c.get('like_count'),
        'timestamp': c.get('timestamp')
    })

ko_df = pd.DataFrame(comments_list)
ko_df.to_csv(DATA_DIR / "yt_comments_ko.csv", index=False, encoding='utf-8-sig')

# 5. '처벌' 키워드가 들어간 댓글만 필터링해서 보기
punish_comments = ko_df[ko_df['text'].str.contains('처벌|구속|수사', na=False)]
print(f"총 {len(ko_df)}개의 댓글 중 규제/처벌 관련 댓글 {len(punish_comments)}개를 찾았습니다.")
print(punish_comments['text'].head())


## 3. Meta 실적 발표문 불러오기

`data/`에 배치한 2025년 Q1~Q3 텍스트 파일을 분기 정보와 함께 하나의 데이터프레임으로 통합합니다.


In [ ]:
# Meta 2025년 Q1~Q3 실적 발표문을 불러옵니다.
files = [
    DATA_DIR / "Q1_2025.txt",
    DATA_DIR / "Q2_2025.txt",
    DATA_DIR / "Q3_2025.txt",
]
data_list = []

for file in files:
    if file.exists():
        with file.open("r", encoding="utf-8") as handle:
            content = handle.read()
            quarter = file.name.split("_")[0]
            data_list.append({"Quarter": quarter, "Content": content})
    else:
        print(f"Warning: File not found at {file}")

meta_df = pd.DataFrame(data_list)

if not meta_df.empty:
    print("메타 데이터 통합 완료!")
    print(meta_df.head())
else:
    print("경고: 파일을 찾을 수 없거나 데이터가 통합되지 않았습니다.")


## 4. 안전·규제 키워드 분석


In [ ]:
# 분석할 키워드 설정
keywords = 'safety|integrity|regulation|compliance|legal|policy'

# 각 분기별 텍스트에서 키워드가 포함된 문장만 추출하는 함수
def extract_safety_sentences(text):
    sentences = text.split('.') # 문장 단위로 분리
    relevant = [s.strip() for s in sentences if any(k in s.lower() for k in keywords.split('|'))]
    return " / ".join(relevant)

meta_df['Safety_Focus'] = meta_df['Content'].apply(extract_safety_sentences)

# 결과 확인
for i, row in meta_df.iterrows():
    print(f"\n[{row['Quarter']} 규제/안전 관련 주요 발언]:")
    print(row['Safety_Focus'][:1000] + "...") # 너무 길면 앞부분만 출력


### 4.1 한국·미국 담론 비교

다음 막대그래프의 값은 원본 Notebook에 포함된 해커톤 시연용 입력값이며, 앞 단계에서 자동 집계한 값이 아닙니다.


In [ ]:
# 2. 가상의 분석 데이터 (이전 단계에서 수집한 빈도수 결과를 여기에 넣으세요)
kr_data = {'처벌/징역': 45, '구속/수사': 32, '개인정보': 15, '표현의자유': 8}
us_data = {'Safety(안전)': 55, 'Integrity(무결성)': 40, 'Investment(투자)': 65, 'Compliance(준수)': 30}

# 3. 그래프 그리기
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 한국 차트 (유튜브 댓글 반응)
ax[0].bar(kr_data.keys(), kr_data.values(), color='salmon')
ax[0].set_title('한국: 처벌 및 수사 중심 반응 (유튜브 댓글)', fontsize=15)
ax[0].set_ylabel('언급 빈도')

# 미국 차트 (메타 실적 발표 발언)
ax[1].bar(us_data.keys(), us_data.values(), color='skyblue')
ax[1].set_title('미국(Meta): 기술 투자 및 안전 중심 전략 (실적 발표)', fontsize=15)
ax[1].set_ylabel('언급 빈도')

plt.tight_layout()
plt.show()


### 4.2 국가별 워드클라우드


In [ ]:
# 3. 데이터 준비 (통합한 데이터프레임에서 텍스트 추출)
# ko_df['text']와 meta_df['Content']가 있다고 가정합니다.
kr_text = " ".join(ko_df['text'].dropna())
us_text = " ".join(meta_df['Content'].dropna())

# 4. 워드클라우드 생성 함수
def make_wordcloud(text, title, color):
    wc = WordCloud(
        font_path=font_path,
        background_color='white',
        colormap=color,
        width=800, height=400
    ).generate(text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(title, fontproperties=fontprop, fontsize=20)
    plt.axis('off')
    plt.show()

# 5. 실행
print("--- 한국 유튜브 댓글 워드클라우드 ---")
make_wordcloud(kr_text, "한국: 규제 및 처벌 중심 키워드", 'Reds')

print("--- 미국 Meta 실적발표 워드클라우드 ---")
make_wordcloud(us_text, "미국(Meta): 안전 및 투자 중심 키워드", 'Blues')


### 4.3 Meta 분기별 키워드 추이


In [ ]:
# 1. 분기별 키워드 빈도 계산
quarters = ['Q1', 'Q2', 'Q3']
safety_counts = []

for q in quarters:
    # 해당 분기 텍스트에서 'safety' 또는 'compliance' 단어 개수 세기
    text = meta_df[meta_df['Quarter'] == q]['Content'].values[0].lower()
    count = text.count('safety') + text.count('compliance') + text.count('integrity')
    safety_counts.append(count)

# 2. 추이 그래프 그리기
plt.figure(figsize=(10, 6))
plt.plot(quarters, safety_counts, marker='o', linestyle='-', color='dodgerblue', linewidth=3)

# 그래프 한글 설정
plt.title('2025년 Meta의 안전/규제 관련 언급 추이', fontproperties=fontprop, fontsize=16)
plt.xlabel('분기 (2025)', fontproperties=fontprop)
plt.ylabel('키워드 언급 횟수', fontproperties=fontprop)
plt.grid(True, linestyle='--')
plt.show()


## 5. 감성 및 부정 반응 키워드 분석


In [ ]:
# 1. 한국어 감성 분석용 사전 (간이 버전)
# 실제 연구 시에는 더 큰 사전을 쓰지만, 여기서는 핵심 키워드로 온도를 측정합니다.
okt = Okt()
def get_kr_sentiment(text):
    # 공포/부정 키워드 vs 기대/긍정 키워드
    neg_words = ['처벌', '구속', '무섭다', '감시', '침해', '악용', '징역']
    pos_words = ['보호', '안전', '당연하다', '필요하다', '찬성', '근절']

    score = 0
    words = okt.nouns(text)
    for word in words:
        if word in neg_words: score -= 1
        if word in pos_words: score += 1
    return score

# 2. 영어(Meta) 감성 분석 (TextBlob 사용)
def get_us_sentiment(text):
    # -1(부정) ~ 1(긍정) 사이의 값을 반환
    return TextBlob(text).sentiment.polarity

# 3. 데이터 적용 (데이터프레임이 준비되어 있어야 함)
# 한국 댓글 감성 측정
ko_df['sentiment_score'] = ko_df['text'].apply(get_kr_sentiment)

# 미국 메타 실적 발표 감성 측정 (Q1, Q2, Q3 통합)
meta_df['sentiment_score'] = meta_df['Content'].apply(get_us_sentiment)

# 4. 결과 시각화 (감정 온도계)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# 한국: 규제 관련 댓글의 감정 분포
ax[0].hist(ko_df['sentiment_score'], bins=10, color='indianred', alpha=0.7)
ax[0].set_title('한국: 규제 뉴스 댓글 감성 분포 (부정적/방어적)', fontproperties=fontprop)
ax[0].set_xlabel('감성 점수 (낮을수록 공포/부정)', fontproperties=fontprop)

# 미국: 메타 경영진의 발언 감성 분포
ax[1].hist(meta_df['sentiment_score'], bins=10, color='seagreen', alpha=0.7)
ax[1].set_title('미국(Meta): 실적 발표 감성 분포 (긍정적/자신감)', fontproperties=fontprop)
ax[1].set_xlabel('감성 점수 (높을수록 투자 의지/자신감)', fontproperties=fontprop)

plt.show()


### 5.1 한국 부정 댓글의 주요 명사


In [ ]:
# 1. 가장 부정적인 댓글 Top 20 추출
negative_comments = ko_df.sort_values(by='sentiment_score').head(20)

# 2. 분노의 핵심 키워드 분석 (Nouns 추출)
all_neg_nouns = []
for text in negative_comments['text']:
    all_neg_nouns.extend(okt.nouns(text))

# 3. 불용어 처리 및 빈도 계산
stop_words = ['진짜', '정말', '이게', '그냥', '뉴스', '생각']
filtered_nouns = [n for n in all_neg_nouns if n not in stop_words and len(n) > 1]

neg_reason_counts = Counter(filtered_nouns).most_common(10)

print("--- 한국인들을 화나게 만드는 규제의 핵심 요인 ---")
for word, count in neg_reason_counts:
    print(f"키워드: {word} (언급 횟수: {count})")


## 6. 인지 상태 라벨링과 전이행렬


In [ ]:
# 한국과 미국의 인지 상태(Node) 정의
def labeling_cognitive_state(text, country='KOR'):
    text = text.lower()
    if country == 'KOR':
        if any(w in text for w in ['처벌', '징역', '구속', '사형']): return '처벌 갈구(Punishment)'
        if any(w in text for w in ['중국', '감시', '통제', '기록']): return '감시 공포(Fear of Surveillance)'
        if any(w in text for w in ['기준', '모호', '억울', '법']): return '제도 불신(Distrust)'
        return '기타/무관심'
    else: # USA (Meta)
        if any(w in text for w in ['invest', 'capex', 'infrastructure']): return '투자 의지(Investment)'
        if any(w in text for w in ['compliance', 'regulation', 'safety']): return '준수 의지(Compliance)'
        if any(w in text for w in ['innovation', 'llama', 'ai']): return '혁신 강조(Innovation)'
        return '기타'

# 데이터에 라벨 적용
ko_df['state'] = ko_df['text'].apply(lambda x: labeling_cognitive_state(x, 'KOR'))
meta_df['state'] = meta_df['Content'].apply(lambda x: labeling_cognitive_state(x, 'USA'))


### 6.1 시간·분기 순서 정렬


In [ ]:
# 한국 데이터 시간순 정렬 (유튜브 댓글 타임스탬프 기준)
ko_df = ko_df.sort_values(by='timestamp')

# 미국 데이터 분기순 정렬
meta_df['quarter_num'] = meta_df['Quarter'].str.replace('Q', '').astype(int)
meta_df = meta_df.sort_values(by='quarter_num')


### 6.2 한국 인지 상태 전이 확률


In [ ]:
def calculate_transition_matrix(df):
    states = df['state'].unique()
    state_to_idx = {state: i for i, state in enumerate(states)}
    n = len(states)

    # 전이 행렬 초기화
    matrix = np.zeros((n, n))

    # 상태 변화 카운트 (t 시점 -> t+1 시점)
    current_states = df['state'].values
    for i in range(len(current_states)-1):
        start_node = current_states[i]
        end_node = current_states[i+1]
        matrix[state_to_idx[start_node]][state_to_idx[end_node]] += 1

    # 확률로 변환 (행의 합이 1이 되도록)
    row_sums = matrix.sum(axis=1)
    transition_prob = np.divide(matrix, row_sums[:, np.newaxis], where=row_sums[:, np.newaxis]!=0)

    return pd.DataFrame(transition_prob, index=states, columns=states)

# 한국 사회의 인지 전이 확률 계산
ko_transition = calculate_transition_matrix(ko_df)
print("--- 한국 사회 인지 전이 확률 행렬 ---")
print(ko_transition)


In [ ]:
def plot_transition_heatmap(matrix, title):
    plt.figure(figsize=(10, 8))
    sns.heatmap(matrix, annot=True, cmap='YlGnBu', fmt='.2f', cbar_kws={'label': '전이 확률'})

    plt.title(f"{title} 인지 전이 확률 지도", fontproperties=fontprop, fontsize=15)
    plt.ylabel('현재 인지 상태 (t)', fontproperties=fontprop)
    plt.xlabel('다음 인지 상태 (t+1)', fontproperties=fontprop)

    # 틱 레이블 한글 설정
    plt.xticks(fontproperties=fontprop)
    plt.yticks(fontproperties=fontprop)
    plt.show()

# 한국 데이터 전이 히트맵 출력
plot_transition_heatmap(ko_transition, "한국 사회")


## 7. 처벌 프레임 가중치 시뮬레이션


In [ ]:
def simulate_digital_twin(punish_frame_weight):
    # 기본 전이 행렬 복사
    sim_matrix = ko_transition.copy()

    # '처벌' 프레임 가중치 조작 (특정 상태로의 전이 확률 강제 상향)
    target_state = '처벌 갈구(Punishment)'
    if target_state in sim_matrix.columns:
        sim_matrix[target_state] = sim_matrix[target_state] * punish_frame_weight
        # 확률의 합이 1이 되도록 재정규화
        sim_matrix = sim_matrix.div(sim_matrix.sum(axis=1), axis=0)

    # 10단계 후의 최종 인지 상태 분포 예측 (마르코프 체인 평형 상태)
    initial_distribution = np.array([1/len(sim_matrix)] * len(sim_matrix))
    current_dist = initial_distribution
    for _ in range(10):
        current_dist = np.dot(current_dist, sim_matrix.values)

    # 결과 시각화
    plt.figure(figsize=(10, 5))
    colors = ['salmon' if s == target_state else 'lightgrey' for s in sim_matrix.index]
    plt.bar(sim_matrix.index, current_dist, color=colors)
    plt.title(f"프레임 조작 후 최종 인지 상태 예측 (처벌 강조 가중치: {punish_frame_weight}x)", fontproperties=fontprop)
    plt.ylabel('예상 점유율', fontproperties=fontprop)
    plt.xticks(fontproperties=fontprop, rotation=45)
    plt.ylim(0, 1)
    plt.show()

# 슬라이더 실행 (가중치를 1.0에서 3.0까지 조절해보세요)
interact(simulate_digital_twin,
         punish_frame_weight=FloatSlider(min=1.0, max=5.0, step=0.5, value=1.0, description='처벌 프레임 ↑'));


## 8. 미국 인지 상태 전이와 국가 비교


In [ ]:
# 미국 (Meta) 인지 전이 확률 계산
us_transition = calculate_transition_matrix(meta_df)
print("--- 미국 (Meta) 인지 전이 확률 행렬 ---")
print(us_transition)


In [ ]:
def compare_heatmaps(ko_matrix, us_matrix):
    fig, ax = plt.subplots(1, 2, figsize=(18, 7))

    # 한국 히트맵
    sns.heatmap(ko_matrix, annot=True, cmap='Reds', fmt='.2f', ax=ax[0])
    ax[0].set_title('KOREA: 인지 전이 지도 (처벌/감시 중심)', fontsize=14)

    # 미국 히트맵
    sns.heatmap(us_matrix, annot=True, cmap='Blues', fmt='.2f', ax=ax[1])
    ax[1].set_title('USA (Meta): 인지 전이 지도 (투자/준수 중심)', fontsize=14)

    plt.tight_layout()
    plt.show()

# 이전 단계에서 계산된 ko_transition, us_transition 사용
compare_heatmaps(ko_transition, us_transition)


## 9. 정리 전 실행 결과 요약

- 마지막 분석 대상 YTN 영상: 댓글 57개 중 처벌·구속·수사 관련 댓글 15개
- 부정 댓글 주요 명사: `처벌` 16회, `페이크` 7회, `시청` 6회, `나라` 6회
- Meta Q1~Q3: 현재 라벨링 규칙에서 모두 투자 의지 상태로 분류되어 자기 전이 확률 1.0

이 값은 정리 전 Notebook에 저장되어 있던 출력의 요약입니다. 유튜브 댓글은 실행 시점에 따라 바뀔 수 있으며, 해커톤 표본과 규칙 기반 모델의 한계를 고려해 해석해야 합니다. 또한 어떤 상태가 마지막 행에서만 등장해 나가는 전이가 없으면 기존 전이행렬 계산식이 해당 행에 미정 값을 남길 수 있습니다.
